In [1]:
import pandas as pd
import numpy as np

In [2]:
df_gdp = pd.read_csv("../data/raw/gdp.csv", skiprows=4)

df_gdp = df_gdp[df_gdp["Country Name"] == "Sri Lanka"]

df_gdp = df_gdp.drop(columns=["Country Code", "Indicator Name", "Indicator Code"])

df_gdp_long = df_gdp.melt(id_vars=["Country Name"], var_name="Year", value_name="GDP")

df_gdp_long = df_gdp_long.rename(columns={"Country Name": "Country"})

df_gdp_long = df_gdp_long.dropna()

df_gdp_long["Year"] = df_gdp_long["Year"].astype(int)

df_gdp_long = df_gdp_long.sort_values("Year").reset_index(drop=True)

df_gdp_long.head()

,Country,Year,GDP
0,Sri Lanka,1960,145.928701
1,Sri Lanka,1961,145.900945
2,Sri Lanka,1962,141.383198
3,Sri Lanka,1963,119.352332
4,Sri Lanka,1964,122.941809


In [3]:
df_imports = pd.read_csv("../data/raw/imports.csv", skiprows=4)

df_imports = df_imports[df_imports["Country Name"] == "Sri Lanka"]

df_imports = df_imports.drop(columns=["Country Code", "Indicator Name", "Indicator Code"])

df_imports_long = df_imports.melt(id_vars=["Country Name"], var_name="Year", value_name="Imports")

df_imports_long = df_imports_long.rename(columns={"Country Name": "Country"})

df_imports_long = df_imports_long.dropna()

df_imports_long["Year"] = df_imports_long["Year"].astype(int)

df_imports_long = df_imports_long.sort_values("Year").reset_index(drop=True)

df_imports_long.head()

,Country,Year,Imports
0,Sri Lanka,2000,4.48
1,Sri Lanka,2001,3.50
2,Sri Lanka,2002,3.33
3,Sri Lanka,2003,4.64
4,Sri Lanka,2004,4.17


In [4]:
df_ewaste = pd.read_csv("../data/raw/ewaste.csv")

df_ewaste = df_ewaste[["Country", "Year", "E_Waste_Generation_Million_Metric_Tons"]]

df_ewaste = df_ewaste.rename(columns={
    "E_Waste_Generation_Million_Metric_Tons": "E_waste_MT"
})

# convert to metric tons
df_ewaste["E_waste_MT"] = df_ewaste["E_waste_MT"] * 1_000_000

# select one country (India) and map to Sri Lanka
df_ewaste = df_ewaste[df_ewaste["Country"] == "India"]
df_ewaste["Country"] = "Sri Lanka"

df_ewaste = df_ewaste.dropna()

df_ewaste = df_ewaste.sort_values("Year").reset_index(drop=True)

df_ewaste.head()

,Country,Year,E_waste_MT
0,Sri Lanka,2015,4100000.0
1,Sri Lanka,2016,4370000.0
2,Sri Lanka,2017,4750000.0
3,Sri Lanka,2018,4860000.0
4,Sri Lanka,2019,4780000.0


In [5]:
df_merge = pd.merge(df_gdp_long, df_imports_long, on=["Country", "Year"])

df_final = pd.merge(df_merge, df_ewaste, on=["Country", "Year"])

df_final = df_final.sort_values("Year").reset_index(drop=True)

df_final.head()

,Country,Year,GDP,Imports,E_waste_MT
0,Sri Lanka,2015,4057.715835,4.22,4100000.0
1,Sri Lanka,2016,4149.191908,4.95,4370000.0
2,Sri Lanka,2017,4398.888281,4.73,4750000.0
3,Sri Lanka,2019,4081.947727,4.64,4780000.0
4,Sri Lanka,2020,3847.601377,5.36,4600000.0


In [6]:
# create full year range
full_years = pd.DataFrame({
    "Year": range(df_final["Year"].min(), df_final["Year"].max() + 1)
})

df_final = pd.merge(full_years, df_final, on="Year", how="left")

# forward fill
df_final = df_final.ffill()

df_final["Country"] = "Sri Lanka"

df_final = df_final.sort_values("Year").reset_index(drop=True)

df_final.head(10)

,Year,Country,GDP,Imports,E_waste_MT
0,2015,Sri Lanka,4057.715835,4.22,4100000.0
1,2016,Sri Lanka,4149.191908,4.95,4370000.0
2,2017,Sri Lanka,4398.888281,4.73,4750000.0
3,2018,Sri Lanka,4398.888281,4.73,4750000.0
4,2019,Sri Lanka,4081.947727,4.64,4780000.0
5,2020,Sri Lanka,3847.601377,5.36,4600000.0
6,2021,Sri Lanka,3996.962400,6.20,4700000.0
7,2022,Sri Lanka,3342.636503,3.01,5040000.0


In [7]:
from sklearn.preprocessing import MinMaxScaler

features = df_final[["GDP", "Imports"]]
target = df_final[["E_waste_MT"]]

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(features)
y_scaled = scaler_y.fit_transform(target)

X_scaled_df = pd.DataFrame(X_scaled, columns=["GDP", "Imports"])
y_scaled_df = pd.DataFrame(y_scaled, columns=["E_waste_MT"])

print(X_scaled_df.head())
print(y_scaled_df.head())

        GDP   Imports
0  0.676997  0.379310
1  0.763601  0.608150
2  1.000000  0.539185
3  1.000000  0.539185
4  0.699938  0.510972
   E_waste_MT
0    0.000000
1    0.287234
2    0.691489
3    0.691489
4    0.723404


In [8]:
print(df_final.head())
print(df_final.columns)

   Year    Country          GDP  Imports  E_waste_MT
0  2015  Sri Lanka  4057.715835     4.22   4100000.0
1  2016  Sri Lanka  4149.191908     4.95   4370000.0
2  2017  Sri Lanka  4398.888281     4.73   4750000.0
3  2018  Sri Lanka  4398.888281     4.73   4750000.0
4  2019  Sri Lanka  4081.947727     4.64   4780000.0
Index(['Year', 'Country', 'GDP', 'Imports', 'E_waste_MT'], dtype='object')


In [9]:
print(df_final.isnull().sum())

Year          0
Country       0
GDP           0
Imports       0
E_waste_MT    0
dtype: int64


In [10]:
print(df_final["Year"].tolist())

[2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]


In [11]:
print(len(df_final))

8


In [12]:
print(X_scaled_df.describe())
print(y_scaled_df.describe())

            GDP   Imports
count  8.000000  8.000000
mean   0.654761  0.539185
std    0.319360  0.286153
min    0.000000  0.000000
25%    0.584127  0.478056
50%    0.688468  0.539185
75%    0.822701  0.640282
max    1.000000  1.000000
       E_waste_MT
count    8.000000
mean     0.570479
std      0.304472
min      0.000000
25%      0.470745
50%      0.664894
75%      0.699468
max      1.000000


In [13]:
print(X_scaled_df.shape)
print(y_scaled_df.shape)

(8, 2)
(8, 1)


In [14]:
# split index (80% approx)
split_index = int(len(df_final) * 0.8)

# split dataset
train = df_final.iloc[:split_index]
test = df_final.iloc[split_index:]

print("Train data:")
print(train)

print("\nTest data:")
print(test)

Train data:
   Year    Country          GDP  Imports  E_waste_MT
0  2015  Sri Lanka  4057.715835     4.22   4100000.0
1  2016  Sri Lanka  4149.191908     4.95   4370000.0
2  2017  Sri Lanka  4398.888281     4.73   4750000.0
3  2018  Sri Lanka  4398.888281     4.73   4750000.0
4  2019  Sri Lanka  4081.947727     4.64   4780000.0
5  2020  Sri Lanka  3847.601377     5.36   4600000.0

Test data:
   Year    Country          GDP  Imports  E_waste_MT
6  2021  Sri Lanka  3996.962400     6.20   4700000.0
7  2022  Sri Lanka  3342.636503     3.01   5040000.0


In [15]:
print(train["Year"].min(), train["Year"].max())
print(test["Year"].min(), test["Year"].max())

2015 2020
2021 2022


In [16]:
from sklearn.preprocessing import MinMaxScaler

# features and target
X_train = train[["GDP", "Imports"]]
X_test = test[["GDP", "Imports"]]

y_train = train[["E_waste_MT"]]
y_test = test[["E_waste_MT"]]

In [17]:
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

In [18]:
X_train_scaled = scaler_X.fit_transform(X_train)
y_train_scaled = scaler_y.fit_transform(y_train)

In [19]:
X_test_scaled = scaler_X.transform(X_test)
y_test_scaled = scaler_y.transform(y_test)

In [20]:
import pandas as pd

print(pd.DataFrame(X_train_scaled, columns=["GDP","Imports"]))
print(pd.DataFrame(y_train_scaled, columns=["E_waste"]))

        GDP   Imports
0  0.381134  0.000000
1  0.547066  0.640351
2  1.000000  0.447368
3  1.000000  0.447368
4  0.425090  0.368421
5  0.000000  1.000000
    E_waste
0  0.000000
1  0.397059
2  0.955882
3  0.955882
4  1.000000
5  0.735294


In [21]:
import numpy as np

def create_sequences(X, y, window_size):
    X_seq = []
    y_seq = []

    for i in range(len(X) - window_size):
        X_seq.append(X[i:i+window_size])
        y_seq.append(y[i+window_size])

    return np.array(X_seq), np.array(y_seq)

In [22]:
window_size = 3

X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train_scaled, window_size)
X_test_seq, y_test_seq = create_sequences(X_test_scaled, y_test_scaled, window_size)

In [23]:
print(X_train_seq.shape)
print(y_train_seq.shape)

(3, 3, 2)
(3, 1)


In [24]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

ModuleNotFoundError: No module named 'tensorflow'

In [25]:
pip install tensorflow

   ---------------------------------------- 0.0/350.9 MB ? eta -:--:--
   ---------------------------------------- 0.5/350.9 MB 2.8 MB/s eta 0:02:06
   ---------------------------------------- 1.6/350.9 MB 5.6 MB/s eta 0:01:03
   ---------------------------------------- 1.6/350.9 MB 5.6 MB/s eta 0:01:03
   ---------------------------------------- 1.6/350.9 MB 5.6 MB/s eta 0:01:03
   ---------------------------------------- 1.8/350.9 MB 1.7 MB/s eta 0:03:25
   ---------------------------------------- 1.8/350.9 MB 1.7 MB/s eta 0:03:25
   ---------------------------------------- 2.1/350.9 MB 1.5 MB/s eta 0:03:58
   ---------------------------------------- 3.1/350.9 MB 1.9 MB/s eta 0:03:00
   ---------------------------------------- 3.1/350.9 MB 1.9 MB/s eta 0:03:00
   ---------------------------------------- 3.1/350.9 MB 1.9 MB/s eta 0:03:00
   ---------------------------------------- 3.7/350.9 MB 1.6 MB/s eta 0:03:40
    --------------------------------------- 4.5/350.9 MB 1.8 MB/s eta 0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
googleapis-common-protos 1.72.0 requires protobuf!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.34.1 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.34.1 which is incompatible.
google-api-core 2.28.1 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.19.5, but you have protobuf 7.34.1 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.34.1 which is incompatible.
proto-plus 1.27.0 requires protobuf<7.0.0,>=3.19.0, but you have protobuf 7.34.1 which is incompatible.

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: D:\so

In [26]:
pip install tensorflow==2.16.1

   ---------------------------------------- 0.0/377.1 MB ? eta -:--:--
   ---------------------------------------- 1.0/377.1 MB 10.1 MB/s eta 0:00:38
   ---------------------------------------- 1.0/377.1 MB 10.1 MB/s eta 0:00:38
   ---------------------------------------- 1.3/377.1 MB 1.9 MB/s eta 0:03:17
   ---------------------------------------- 2.1/377.1 MB 2.6 MB/s eta 0:02:27
   ---------------------------------------- 2.1/377.1 MB 2.6 MB/s eta 0:02:27
   ---------------------------------------- 2.1/377.1 MB 2.6 MB/s eta 0:02:27
   ---------------------------------------- 3.4/377.1 MB 2.4 MB/s eta 0:02:36
   ---------------------------------------- 3.4/377.1 MB 2.4 MB/s eta 0:02:36
   ---------------------------------------- 3.4/377.1 MB 2.4 MB/s eta 0:02:36
   ---------------------------------------- 4.2/377.1 MB 2.0 MB/s eta 0:03:09
    --------------------------------------- 5.8/377.1 MB 2.5 MB/s eta 0:02:27
    --------------------------------------- 5.8/377.1 MB 2.5 MB/s eta

  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.9 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: D:\softwares\Files\Python\python.exe -m pip install --upgrade pip


In [1]:
import tensorflow as tf
print(tf.__version__)

AttributeError: module 'tensorflow' has no attribute '__version__'

In [1]:
import tensorflow as tf
print(tf.__version__)

2.16.1


In [2]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

In [3]:
model = Sequential()

model.add(LSTM(50, activation='relu', input_shape=(3, 2)))
model.add(Dense(1))

model.summary()

D:\Work\04_Y4S1\03_RP\Research\new component 2\project\e_waste_project\ml\venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                          │ (None, 50)                  │          10,600 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 1)                   │              51 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 10,651 (41.61 KB)

 Trainable params: 10,651 (41.61 KB)

 Non-trainable params: 0 (0.00 B)

In [37]:
model.compile(optimizer='adam', loss='mse')

In [38]:
history = model.fit(
    X_train_seq, y_train_seq,
    epochs=30,
    verbose=1
)

Epoch 1/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step - loss: 0.1003
Epoch 2/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 164ms/step - loss: 0.0915
Epoch 3/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 179ms/step - loss: 0.0832
Epoch 4/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 167ms/step - loss: 0.0753
Epoch 5/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 166ms/step - loss: 0.0679
Epoch 6/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 170ms/step - loss: 0.0610
Epoch 7/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 170ms/step - loss: 0.0547
Epoch 8/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 166ms/step - loss: 0.0489
Epoch 9/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 174ms/step - loss: 0.0438
Epoch 10/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 180ms/step - loss: 0.0394
Epoch 11/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 224ms/step - loss: 0.0358
Epoch 12/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 159ms/step - loss: 0.0328
Epoch 13/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 152ms/step - loss: 0.0306
Epoch 14/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 163ms/step - loss: 0.0291
Epoch 15/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 160ms/step - loss: 0.0283
Epoch 16/30
1/1 ━━━━━━

In [39]:
print(X_train_seq.shape)
print(y_train_seq.shape)

(3, 3, 2)
(3, 1)


In [40]:
y_pred = model.predict(X_train_seq)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 611ms/step


In [41]:
y_pred_actual = scaler_y.inverse_transform(y_pred)
y_actual = scaler_y.inverse_transform(y_train_seq)

In [42]:
for i in range(len(y_pred_actual)):
    print(f"Predicted: {y_pred_actual[i][0]:.2f} | Actual: {y_actual[i][0]:.2f}")

Predicted: 4623138.50 | Actual: 4750000.00
Predicted: 4779196.00 | Actual: 4780000.00
Predicted: 4743537.00 | Actual: 4600000.00


In [43]:
from tensorflow.keras.layers import Bidirectional

In [44]:
model = Sequential()

model.add(Bidirectional(LSTM(50, activation='relu'), input_shape=(3, 2)))
model.add(Dense(1))

model.summary()

D:\Work\04_Y4S1\03_RP\Research\new component 2\project\e_waste_project\ml\venv\Lib\site-packages\keras\src\layers\rnn\bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)        │ (None, 100)                 │          21,200 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │             101 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 21,301 (83.21 KB)

 Trainable params: 21,301 (83.21 KB)

 Non-trainable params: 0 (0.00 B)

In [45]:
model.compile(optimizer='adam', loss='mse')

history = model.fit(
    X_train_seq, y_train_seq,
    epochs=30,
    verbose=1
)

Epoch 1/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 8s 8s/step - loss: 1.0525
Epoch 2/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 144ms/step - loss: 1.0258
Epoch 3/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step - loss: 0.9998
Epoch 4/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 130ms/step - loss: 0.9744
Epoch 5/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 155ms/step - loss: 0.9498
Epoch 6/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 129ms/step - loss: 0.9259
Epoch 7/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 140ms/step - loss: 0.9027
Epoch 8/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 138ms/step - loss: 0.8805
Epoch 9/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 139ms/step - loss: 0.8585
Epoch 10/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 152ms/step - loss: 0.8368
Epoch 11/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 158ms/step - loss: 0.8153
Epoch 12/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 176ms/step - loss: 0.7939
Epoch 13/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 141ms/step - loss: 0.7729
Epoch 14/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 179ms/step - loss: 0.7522
Epoch 15/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 178ms/step - loss: 0.7319
Epoch 16/30
1/1 ━━━━━━